In [1]:
# ============================================================
# Kaggle offline inference for Pixels-to-Predictions
# SmolVLM-500M-Instruct + LoRA adapter
# Output: /kaggle/working/submission.csv
# ============================================================

import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import ast
import json
import gc
import random
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from tqdm.auto import tqdm

from transformers import AutoProcessor, Idefics3ForConditionalGeneration
from peft import PeftModel


# ============================================================
# 0) Basic config
# ============================================================

SEED = 3407
SAFE_IMAGE_SIZE = 512
MAX_SEQ_LENGTH = 1024
GEN_MAX_NEW_TOKENS = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
elif torch.cuda.is_available():
    compute_dtype = torch.float16
else:
    compute_dtype = torch.float32

print("device:", device)
print("compute_dtype:", compute_dtype)


# ============================================================
# 1) Locate Kaggle input folders
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")
assert KAGGLE_INPUT.exists(), "Missing /kaggle/input. Are you running on Kaggle?"

def find_file_under(root: Path, filename: str):
    matches = list(root.rglob(filename))
    if not matches:
        return None
    # Prefer competition-like paths.
    matches = sorted(matches, key=lambda p: len(str(p)))
    return matches[0]

TEST_CSV = find_file_under(KAGGLE_INPUT, "test.csv")
SAMPLE_SUBMISSION_CSV = find_file_under(KAGGLE_INPUT, "sample_submission.csv")

assert TEST_CSV is not None, "Could not find test.csv under /kaggle/input"
assert SAMPLE_SUBMISSION_CSV is not None, "Could not find sample_submission.csv under /kaggle/input"

COMP_ROOT = TEST_CSV.parent
print("TEST_CSV:", TEST_CSV)
print("SAMPLE_SUBMISSION_CSV:", SAMPLE_SUBMISSION_CSV)
print("COMP_ROOT:", COMP_ROOT)


def looks_like_base_model_dir(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / "config.json").exists()
        and (
            (p / "model.safetensors").exists()
            or any(p.glob("model-*.safetensors"))
            or (p / "pytorch_model.bin").exists()
            or any(p.glob("pytorch_model-*.bin"))
        )
    )

def looks_like_adapter_dir(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / "adapter_config.json").exists()
        and (
            (p / "adapter_model.safetensors").exists()
            or (p / "adapter_model.bin").exists()
        )
    )

base_candidates = [p for p in KAGGLE_INPUT.rglob("*") if looks_like_base_model_dir(p)]
adapter_candidates = [p for p in KAGGLE_INPUT.rglob("*") if looks_like_adapter_dir(p)]

print("\nBase model candidates:")
for p in base_candidates:
    print(" -", p)

print("\nAdapter candidates:")
for p in adapter_candidates:
    print(" -", p)

assert len(base_candidates) > 0, (
    "Could not find base model folder. "
    "You need to add the Kaggle Dataset containing SmolVLM-500M-Instruct base model."
)
assert len(adapter_candidates) > 0, (
    "Could not find LoRA adapter folder. "
    "You need to add the Kaggle Dataset containing final_adapter_after_training."
)

# If there are multiple candidates, prefer names containing smolvlm and final_adapter.
BASE_MODEL_DIR = sorted(
    base_candidates,
    key=lambda p: (
        "smolvlm" not in str(p).lower(),
        len(str(p))
    )
)[0]

ADAPTER_DIR = sorted(
    adapter_candidates,
    key=lambda p: (
        "final_adapter" not in str(p).lower(),
        "epoch" not in str(p).lower(),
        len(str(p))
    )
)[0]

print("\nSelected BASE_MODEL_DIR:", BASE_MODEL_DIR)
print("Selected ADAPTER_DIR:", ADAPTER_DIR)


# ============================================================
# 2) Load raw test data
# ============================================================

test_df = pd.read_csv(TEST_CSV)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)

test_df.columns = [str(c).strip() for c in test_df.columns]
sample_submission_df.columns = [str(c).strip() for c in sample_submission_df.columns]

print("\ntest_df:", test_df.shape)
print("sample_submission_df:", sample_submission_df.shape)
print("test columns:", test_df.columns.tolist())
print("sample_submission columns:", sample_submission_df.columns.tolist())


# ============================================================
# 3) Preprocessing helpers matching training notebook
# ============================================================

IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]

def safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_optional_text(x) -> str:
    s = safe_str(x).replace("\r", " ").replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in {"", "nan", "none", "null", "na", "n/a"}:
        return ""
    return s

def parse_choices(x) -> List[str]:
    if isinstance(x, list):
        raw = x
    else:
        s = safe_str(x).strip()
        if not s:
            return []
        raw = None
        for parser in (json.loads, ast.literal_eval):
            try:
                candidate = parser(s)
                if isinstance(candidate, list):
                    raw = candidate
                    break
            except Exception:
                pass
        if raw is None:
            raw = [s]

    out = []
    for c in raw:
        cs = normalize_optional_text(c)
        if cs:
            out.append(cs)
    return out

def build_prompt(row: pd.Series) -> str:
    question = normalize_optional_text(row.get("question", ""))
    hint = normalize_optional_text(row.get("hint", ""))
    lecture = normalize_optional_text(row.get("lecture", ""))
    task = normalize_optional_text(row.get("task", ""))
    grade = normalize_optional_text(row.get("grade", ""))
    subject = normalize_optional_text(row.get("subject", ""))
    topic = normalize_optional_text(row.get("topic", ""))
    category = normalize_optional_text(row.get("category", ""))
    skill = normalize_optional_text(row.get("skill", ""))

    choices = row.get("choices_parsed", None)
    if not isinstance(choices, list):
        choices = parse_choices(row.get("choices", "[]"))

    formatted_choices = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    meta = []
    for name, val in [
        ("Task", task),
        ("Grade", grade),
        ("Subject", subject),
        ("Topic", topic),
        ("Category", category),
        ("Skill", skill),
    ]:
        if val:
            meta.append(f"{name}: {val}")

    pieces = ["You are solving a multiple-choice science question using the image and text context."]
    if meta:
        pieces.append("\n".join(meta))
    if lecture:
        pieces.append(f"Lecture/Context:\n{lecture}")
    if hint:
        pieces.append(f"Hint:\n{hint}")
    pieces.append(f"Question:\n{question}")
    pieces.append(f"Choices:\n{formatted_choices}")
    pieces.append("Answer with only the integer index of the correct choice, such as 0, 1, 2, or 3.")
    return "\n\n".join(pieces)


# ============================================================
# 4) Robust image resolver
# ============================================================

all_image_files = []
for base in [COMP_ROOT, KAGGLE_INPUT]:
    if base.exists():
        all_image_files.extend([
            p.resolve()
            for p in base.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ])

all_image_files = sorted(set(all_image_files))

image_name_to_paths: Dict[str, List[str]] = {}
image_stem_to_paths: Dict[str, List[str]] = {}

for p in all_image_files:
    image_name_to_paths.setdefault(p.name, []).append(str(p))
    image_stem_to_paths.setdefault(p.stem, []).append(str(p))

print(f"\nIndexed {len(all_image_files)} images.")
print("First few images:")
for p in all_image_files[:5]:
    print(" -", p)

def choose_split_match(paths: List[str], split: str = "test") -> Optional[str]:
    if not paths:
        return None
    split_matches = [m for m in paths if f"/{split}/" in m.replace("\\", "/")]
    return split_matches[0] if split_matches else paths[0]

def resolve_image_path(row: pd.Series, split: str = "test") -> Optional[str]:
    raw_candidates = [
        row.get("image_path", ""),
        row.get("image", ""),
        row.get("img_path", ""),
        row.get("filename", ""),
        row.get("file_name", ""),
    ]

    ex_id = normalize_optional_text(row.get("id", ""))
    candidates = []

    for raw_value in raw_candidates:
        raw = normalize_optional_text(raw_value)
        if not raw:
            continue

        rp = Path(raw)
        candidates += [
            rp,
            COMP_ROOT / raw,
            COMP_ROOT / "images" / raw,
            COMP_ROOT / "images" / split / rp.name,
            COMP_ROOT / "images" / "images" / split / rp.name,
            KAGGLE_INPUT / raw,
        ]

        if rp.name in image_name_to_paths:
            return choose_split_match(image_name_to_paths[rp.name], split)
        if rp.stem in image_stem_to_paths:
            return choose_split_match(image_stem_to_paths[rp.stem], split)

    if ex_id:
        if ex_id in image_stem_to_paths:
            return choose_split_match(image_stem_to_paths[ex_id], split)

        for ext in IMAGE_EXTS:
            candidates += [
                COMP_ROOT / f"{ex_id}{ext}",
                COMP_ROOT / split / f"{ex_id}{ext}",
                COMP_ROOT / "images" / f"{ex_id}{ext}",
                COMP_ROOT / "images" / split / f"{ex_id}{ext}",
                COMP_ROOT / "images" / "images" / split / f"{ex_id}{ext}",
            ]

    for c in candidates:
        try:
            if c.exists() and c.is_file():
                return str(c.resolve())
        except Exception:
            pass

    return None


test_df["choices_parsed"] = test_df["choices"].map(parse_choices) if "choices" in test_df.columns else [[] for _ in range(len(test_df))]
test_df["num_choices"] = test_df["choices_parsed"].map(len)
test_df["prompt"] = test_df.apply(build_prompt, axis=1)
test_df["resolved_image_path"] = test_df.apply(lambda r: resolve_image_path(r, "test"), axis=1)

missing = test_df["resolved_image_path"].isna().sum()
print("Missing test images:", missing, "/", len(test_df))

if missing > 0:
    print(test_df[test_df["resolved_image_path"].isna()].head())
    raise RuntimeError("Some test images could not be resolved. Fix image path resolver before submission.")

print("\nPrepared test_df:")
print(test_df[["id", "num_choices", "resolved_image_path"]].head())


# ============================================================
# 5) Processor / model helpers
# ============================================================

SAFE_IMAGE_KWARGS = {
    "size": {"longest_edge": SAFE_IMAGE_SIZE},
    "max_image_size": {"longest_edge": SAFE_IMAGE_SIZE},
}

def fix_idefics3_processor_image_size(processor, safe_size=SAFE_IMAGE_SIZE):
    ip = getattr(processor, "image_processor", None)
    if ip is None:
        return processor

    safe_dict = {"longest_edge": int(safe_size)}

    try:
        ip.size = dict(safe_dict)
    except Exception as e:
        print("Could not set image_processor.size:", repr(e))

    try:
        ip.max_image_size = dict(safe_dict)
    except Exception as e:
        print("Could not set image_processor.max_image_size:", repr(e))

    print("Patched image processor:")
    print(" image_processor.size =", getattr(ip, "size", None))
    print(" image_processor.max_image_size =", getattr(ip, "max_image_size", None))
    return processor

def safe_processor_call(processor, **kwargs):
    try:
        return processor(
            **kwargs,
            **SAFE_IMAGE_KWARGS,
        )
    except TypeError:
        return processor(**kwargs)

def make_messages(prompt: str, answer=None):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    if answer is not None:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": str(answer)}],
        })
    return messages

def apply_template(processor, messages, add_generation_prompt=False):
    try:
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=add_generation_prompt,
            tokenize=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=add_generation_prompt,
        )

def strip_to_model_inputs(batch):
    allowed = {
        "input_ids",
        "attention_mask",
        "pixel_values",
        "pixel_attention_mask",
        "image_hidden_states",
    }
    return {k: v for k, v in batch.items() if k in allowed}

def extract_choice_index(text: str, num_choices: int = 10) -> int:
    text = str(text).strip()
    matches = re.findall(r"(?<!\d)[0-9](?!\d)", text)
    if not matches:
        matches = re.findall(r"[-+]?\d+", text)

    for m in matches:
        try:
            val = int(m)
            if 0 <= val < max(1, int(num_choices)):
                return val
        except Exception:
            pass

    return 0


# ============================================================
# 6) Load base model + LoRA adapter
# ============================================================

processor_dir = ADAPTER_DIR if (ADAPTER_DIR / "preprocessor_config.json").exists() else BASE_MODEL_DIR

processor = AutoProcessor.from_pretrained(
    processor_dir,
    local_files_only=True,
)
processor = fix_idefics3_processor_image_size(processor, safe_size=SAFE_IMAGE_SIZE)

if hasattr(processor, "tokenizer") and processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = Idefics3ForConditionalGeneration.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    local_files_only=True,
)
base_model.to(device)
base_model.config.use_cache = True

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True,
)
model.to(device)
model.eval()

print("✅ Loaded base model and LoRA adapter.")


# ============================================================
# 7) Generate predictions
# ============================================================

@torch.no_grad()
def predict_one(row):
    img = Image.open(row["resolved_image_path"]).convert("RGB")

    messages = make_messages(row["prompt"], answer=None)
    text = apply_template(processor, messages, add_generation_prompt=True)

    inputs = safe_processor_call(
        processor,
        text=[text],
        images=[img],
        return_tensors="pt",
        padding=True,
    )

    inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    out = model.generate(
        **strip_to_model_inputs(inputs),
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]

    raw = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
    pred = extract_choice_index(raw, num_choices=int(row.get("num_choices", 10)))

    return pred, raw


pred_rows = []

for i in tqdm(range(len(test_df)), desc="Predicting test"):
    row = test_df.iloc[i]
    pred, raw = predict_one(row)
    pred_rows.append({
        "id": row["id"],
        "answer": int(pred),
        "raw_generation": raw,
    })

pred_df = pd.DataFrame(pred_rows)
print(pred_df.head())


# ============================================================
# 8) Build submission.csv
# ============================================================

submission = sample_submission_df.copy()

id_col = "id" if "id" in submission.columns else submission.columns[0]
answer_col = "answer" if "answer" in submission.columns else submission.columns[-1]

pred_map = dict(zip(pred_df["id"], pred_df["answer"]))

submission[answer_col] = (
    submission[id_col]
    .map(pred_map)
    .fillna(0)
    .astype(int)
)

submission_path = Path("/kaggle/working/submission.csv")


submission.to_csv(submission_path, index=False)


print("✅ Saved:", submission_path)

print(submission.head())
print("Submission shape:", submission.shape)

device: cuda
compute_dtype: torch.bfloat16
TEST_CSV: /kaggle/input/competitions/pixels-to-predictions/test.csv
SAMPLE_SUBMISSION_CSV: /kaggle/input/competitions/pixels-to-predictions/sample_submission.csv
COMP_ROOT: /kaggle/input/competitions/pixels-to-predictions

Base model candidates:
 - /kaggle/input/datasets/chenjunjiejaywang/smolvlm/smolvlm_500m_instruct_base_offline

Adapter candidates:
 - /kaggle/input/datasets/chenjunjiejaywang/adapter/final_adapter_after_training

Selected BASE_MODEL_DIR: /kaggle/input/datasets/chenjunjiejaywang/smolvlm/smolvlm_500m_instruct_base_offline
Selected ADAPTER_DIR: /kaggle/input/datasets/chenjunjiejaywang/adapter/final_adapter_after_training

test_df: (1008, 13)
sample_submission_df: (1008, 2)
test columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']
sample_submission columns: ['id', 'answer']

Indexed 5165 images.
First few images:
 - /kaggle/input/competit

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

✅ Loaded base model and LoRA adapter.


Predicting test:   0%|          | 0/1008 [00:00<?, ?it/s]

           id  answer raw_generation
0  test_01750       1              1
1  test_00128       2              2
2  test_02891       0              0
3  test_02425       3              3
4  test_00930       1              1
✅ Saved: /kaggle/working/submission.csv
           id  answer
0  test_01750       1
1  test_00128       2
2  test_02891       0
3  test_02425       3
4  test_00930       1
Submission shape: (1008, 2)
